### Analisis exploratorio de los datos obtenidos de los extractos bancarios

Comentario: *NO* se muestran los plots debido a la sensibilidad de los datos financieros. 
La notebook se deja para demostrar todo el procedimiento de analisis de los datos para la toma de decisiones

In [1]:
import sys

print(sys.executable)
print(sys.version)

/usr/local/bin/python
3.13.15 (main, Sep  1 2026, 00:07:39) [GCC 14.2.0]


In [2]:
import pandas as pd
import re # Permite trabajar con expresiones regulares
import matplotlib.pyplot as plt
import os
import numpy as np
print("Librerias ok")

Librerias ok


In [3]:
import sys
import pandas
import numpy
import matplotlib

print("Python:", sys.version)
print("Python ejecutable:", sys.executable)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("Matplotlib:", matplotlib.__version__)

Python: 3.13.15 (main, Sep  1 2026, 00:07:39) [GCC 14.2.0]
Python ejecutable: /usr/local/bin/python
Pandas: 2.2.3
NumPy: 2.1.3
Matplotlib: 3.9.2


In [2]:
# Se abre el csv generado en la notebook "movimientos_bancarios"
df = pd.read_csv("D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/processed/movimientos_bancarios.csv")


In [3]:
# Vemos que tiene
print(df.info())
#En este caso no nos importa que haya nans en credito y debito, todo lo contrario esta bien

<class 'pandas.DataFrame'>
RangeIndex: 4558 entries, 0 to 4557
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   fecha            4558 non-null   str    
 1   comprobante      2360 non-null   float64
 2   movimiento       4558 non-null   str    
 3   debito           4558 non-null   float64
 4   credito          4558 non-null   float64
 5   saldo_en_cuenta  4558 non-null   float64
 6   archivo_origen   4558 non-null   str    
 7   mes              4558 non-null   str    
 8   categoria        4558 non-null   str    
dtypes: float64(4), str(5)
memory usage: 320.6 KB
None


In [ ]:
#Descripcion del dataset
print(df.describe())

In [ ]:
#Vemos las columnas
print(df.columns)

Index(['Unnamed: 0', 'Fecha', 'Comprobante', 'Movimiento', 'Debito', 'Credito',
       'Saldo en cuenta', 'archivo_origen', 'Mes', 'Categoria'],
      dtype='str')


In [64]:
#Vemos primeras filas
print(df.head(2))

   Unnamed: 0      Fecha  Comprobante  \
0           0 2022-01-05      58472.0   
1           1 2022-01-05   21293998.0   

                                          Movimiento   Debito  Credito  \
0  Debito automatico Bbva seguros -00000000070100...    275.4      0.0   
1  Debito transf. online banking emp A josefina u...  60000.0      0.0   

   Saldo en cuenta archivo_origen      Mes                  Categoria  
0        270159.59  01-2022_banco  2022-01        Debitos automaticos  
1        210159.59  01-2022_banco  2022-01  Transferencias realizadas  


In [ ]:
#Seteamos la faecha con el tipo datetime para poder hacer bien los plots
df["Fecha"] = pd.to_datetime(df["Fecha"],format="%d/%m/%y")
print(df["Fecha"].dtype)
print(df["Fecha"])

datetime64[us]
0      2022-01-05
1      2022-01-05
2      2022-01-05
3      2022-01-07
4      2022-01-07
          ...    
4553   2025-11-27
4554   2025-11-27
4555   2025-11-27
4556   2025-11-27
4557   2025-11-28
Name: Fecha, Length: 4558, dtype: datetime64[us]


In [ ]:
#Vemos el periodo de fechas
fecha_inicio = df["Fecha"].min()
fecha_fin = df["Fecha"].max()
print(f"Período analizado: {fecha_inicio:%d/%m/%Y} - {fecha_fin:%d/%m/%Y}")

Período analizado: 05/01/2022 - 30/06/2026


In [ ]:
#Descricpion del periodo de datos
print("Cantidad de días:", (fecha_fin - fecha_inicio).days + 1)
print("Cantidad de meses:", df["Fecha"].dt.to_period("M").nunique())
print("Cantidad de años:", df["Fecha"].dt.year.nunique())

Cantidad de días: 1638
Cantidad de meses: 52
Cantidad de años: 5


In [ ]:
# Ordenar los datos segun la fecha porque sino queda muy feo el plot
df = df.sort_values("Fecha")

In [ ]:
# Evolución del saldo con movimientos diarios
plt.figure(figsize=(6,4))
plt.plot(df["Fecha"], df["Saldo en cuenta"], linewidth=2, color="#bcbddc")
plt.title("Evolución del saldo bancario con movimientos diarios",fontsize=10)
plt.xlabel("Fecha", fontsize=8)
plt.xticks(rotation=90, fontsize=6)
plt.ylabel("Saldo ($)", fontsize=8)
# plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Evolución del saldo con movimientos mensuales
# Primero se agrupa la info de forma mensual considerando el saldo inicial/total del mes
df["Mes"] = df["Fecha"].dt.to_period("M")
saldo_mensual = (df.sort_values("Fecha").groupby("Mes", as_index=False).last())
saldo_mensual["Mes"] = saldo_mensual["Mes"].astype(str)

#Plot
plt.figure(figsize=(8,4))
plt.plot(saldo_mensual["Mes"], saldo_mensual["Saldo en cuenta"],marker="o",linewidth=2, color="#addd8e")
plt.title("Evolución del saldo mensual")
plt.xlabel("Mes")
plt.ylabel("Saldo de cierre ($)")
plt.xticks(rotation=90, fontsize=9)
# plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# "Flujo mensual creditos vs debitos
# Se crea otra vez la columna por las dudas
df["Mes"] = df["Fecha"].dt.to_period("M").astype(str)
# Como el saldo es acumulativo, lo correcto es tomar el último saldo de cada me
flujo_mensual = (df.groupby("Mes", as_index=False).agg({"Debito": "sum", "Credito": "sum"}))
# Como tenemos NA en debitos y creditos se transforman en numericas
df["Debito"] = pd.to_numeric(df["Debito"], errors="coerce").fillna(0)
df["Credito"] = pd.to_numeric(df["Credito"], errors="coerce").fillna(0)

##plot
x = np.arange(len(flujo_mensual))
ancho = 0.4
plt.figure(figsize=(8,4))
plt.bar(x - ancho/2,flujo_mensual["Credito"], width=ancho,label="Créditos")
plt.bar(x + ancho/2, flujo_mensual["Debito"], width=ancho, label="Débitos" )
plt.xticks(x,flujo_mensual["Mes"],rotation=45, fontsize=9)
plt.title("Flujo mensual de dinero")
plt.xlabel("Mes")
plt.ylabel("Monto ($)")
plt.legend()
# plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# flujo neto (Creditos − Débitos)
flujo_mensual["Flujo Neto"] = (flujo_mensual["Credito"] - flujo_mensual["Debito"])
#Plot
plt.figure(figsize=(8,4))
plt.plot(flujo_mensual["Mes"], flujo_mensual["Flujo Neto"], marker="o", linewidth=2, color="#dd1c77")
plt.axhline(0, linestyle="--", color="#c994c7")
plt.title("Flujo neto mensual")
plt.xlabel("Mes")
plt.ylabel("Créditos - Débitos ($)")
plt.xticks(rotation=45, fontsize=9)
# plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [54]:
#Se hace analisis segun el tipo de concepto:
def clasificar_movimiento(texto):

    texto = str(texto).upper()

    if "TRANSFERENCIA REALIZADA" in texto or "DEBITO TRANSF." in texto:
        return "Transferencias realizadas"

    elif "TRANSFERENCIA RECIBIDA" in texto or "TRANSF RECIBIDA" in texto:
        return "Transferencias recibidas"

    elif "COMISION" in texto:
        return "Comisiones"

    elif "SIRCREB" in texto or "IMPUESTOS" in texto:
        return "Impuestos"

    elif "CHEQUE" in texto or " CH " in texto or "DEPOSITO ECHEQ" in texto or "DEPOSITO E-CHEQ" in texto:
        return "Cheques"

    elif "DEBITO AUTOMATICO" in texto:
        return "Debitos automaticos"

    elif "EXTRACCION" in texto:
        return "Extracciones"

    elif "DEPOSITO" in texto:
        return "Depositos"

    elif "INTERES" in texto:
        return "Intereses"

    elif "IVA 21% " in texto:
            return "Iva 21%"

    elif "PAGO DE SERVICIOS IMP. AFIP" in texto:
        return "Autonomos"

    else:
        return "Otros"
df["Categoria"] = df["Movimiento"].apply(clasificar_movimiento)
print(df["Categoria"].unique())
df.to_csv("D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/processed/movimientos_bancarios.csv")

<StringArray>
[      'Debitos automaticos', 'Transferencias realizadas',
                     'Otros',                'Comisiones',
  'Transferencias recibidas',                 'Depositos',
                   'Cheques',                 'Intereses',
                 'Impuestos',                   'Iva 21%']
Length: 10, dtype: str


In [ ]:
# Mostramos cuanto se gasto con todos los debitos
#debitos = (df[df["Debito"] != "NA"].groupby("Categoria")["Debito"].sum().sort_values(ascending=False))
debitos = (df[df["Debito"] != 0].groupby("Categoria")["Debito"].sum().sort_values(ascending=False))
#Plot
plt.figure(figsize=(6,4))
plt.bar(debitos.index, debitos.values, color="#31a354")
plt.title("Débitos por categoría")
plt.xlabel("Categoría")
plt.ylabel("Monto ($)")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Mostramos cuanto se gasto con todos los creditos
#creditos = (df[df["Credito"] != "NA"].groupby("Categoria")["Credito"] .sum().sort_values(ascending=False))
creditos = (df[df["Credito"] != 0].groupby("Categoria")["Credito"] .sum().sort_values(ascending=False))
#plot
plt.figure(figsize=(6,4))
plt.bar(creditos.index, creditos.values, color="#fc9272")
plt.title("Créditos por categoría")
plt.xlabel("Categoría")
plt.ylabel("Monto ($)")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Hay movimientos duplicados?

In [ ]:
#Hay faltante de datos

Conclusion:
- Se procesaron automáticamente los extractos bancarios mediante Python y pdfplumber.
- Se validó la consistencia de todos los movimientos comprobando la igualdad entre saldo inicial, débitos, créditos y saldo final.
- Las transferencias representan la principal fuente tanto de ingresos como de egresos de la empresa.
- Los impuestos y las comisiones constituyen gastos recurrentes, aunque de menor magnitud que las transferencias.
- El análisis temporal permite identificar meses de mayor actividad financiera y evaluar la evolución del flujo de caja.